# 00 · Choosing the example, and binding at PICALM exon 13 (panels a, b)

**Case Study 3, QKI and two PICALM exons.** This notebook looks for an RBP that binds next to an exon whose splicing changes when that RBP is knocked down. It uses RBP-ARK's Skipper 2.0.0 data, favoring ENCORE and ENCORE2 eCLIP, and finds two PICALM cassette exons that QKI regulates in opposite directions, binding on opposite sides. Run the notebooks in this order:

| Notebook | What it does | Output |
| --- | --- | --- |
| **00_screen_and_locus** (this one) | Panels a, b: screens 18 RBPs for ENCORE/ENCORE2 binding next to exons that respond to knockdown, then shows the QKI windows at PICALM's 150-nt and 24-nt exons | `results/screen_candidates.tsv`, `figures/data/picalm_locus.json` |
| 01_download_shrna_bams | Downloads the 8 ENCODE QKI knockdown and control RNA-seq BAMs (K562 and HepG2), plus ENCODE's rMATS and DESeq2 outputs | `<data>/downloads/shrna_bams_hg19/`, `<data>/downloads/encode_reference/` |
| 02_run_rmats | Runs rMATS on those BAMs (2 knockdown vs control comparisons) | `<data>/rmats/rerun/<knockdown>/` |
| 03_panel_c_sashimi | Panel c: PICALM exon 13 and 24-nt exon sashimi plots and PSI per replicate | `figures/panel_c/` |
| 04_eclip_rbp_maps | Panels d, e: downloads the 7 eCLIP BAMs of the two QKI datasets in the figure's splice maps from RBP-ARK and builds those maps, with PICALM's own traces | `<data>/rbp_maps/`, `figures/panel_de/` |
| 05_window_maps (optional) | Splice maps from RBP-ARK's windows table instead of reads: no BAMs needed | `results/window_map_*.tsv` |

**Folders.** Small results and figures go in this case study folder. Downloads and other large files go in `<data>`: `CASE_STUDY_DATA`, by default `data/` in the case study folder. All six notebooks together download about 45 GB.

**This notebook.**

| Step | What it does |
| --- | --- |
| 1 | RBP-ARK experiment catalog: PASS ENCORE/ENCORE2 experiments whose RBP has ENCODE knockdown data |
| 2 | ENCODE knockdown rMATS (skipped exons, hg19): events changed in the same direction in K562 and HepG2 |
| 3 | Lift those events to GRCh38 |
| 4 | RBP-ARK windows of the same RBP within 500 nt of each exon; rank by ENCORE/ENCORE2 support |
| 5 | Panel b: QKI windows at PICALM exon 13 and the 24-nt exon, the RNA under them, and the GENCODE v40 transcripts |

**Data and time.** The windows table is ~80 MB; each rMATS tar is 3-8 MB (18 RBPs × 2 cell lines). About 10-20 minutes the first time.

**RBP-ARK API.** `RBPARK_API` defaults to `https://rbp-ark.com`, the RBP-ARK production site, which serves the public data API.

## Accessing RBP-ARK data through Globus

RBP-ARK keeps every Skipper output on one Globus collection. Public files can be downloaded straight over HTTPS with no login; bulk copies go through Globus.

| | |
| --- | --- |
| Globus collection | **RBP-ARK**, `9cd9be17-cbed-4b34-ab76-704b2fd6ac7b` ([open in the Globus web app](https://app.globus.org/file-manager?origin_id=9cd9be17-cbed-4b34-ab76-704b2fd6ac7b&origin_path=%2F)) |
| HTTPS address of the same files | `https://g-f8db23.0ed28.75bc.data.globus.org/<path>` |

**Where things are.** `<version>/<run>` is the Skipper version and run folder. Since November 2026 every ENCODE3, ENCORE and ENCORE2 dataset on rbp-ark.com (gene search, dataset pages, BDBags) is Skipper `2.0.0/v2`, and so are these notebooks. Earlier runs (`1.99.0/v1`, `1.100.0/v1`) remain on the collection.

| Path on the collection | Contents |
| --- | --- |
| `/encode3/eclip/Skipper/<version>/<run>/output/` | ENCODE3 eCLIP datasets, named `<RBP>_<cell>_ENCSR…` |
| `/encore2/eclip/Skipper/<version>/<run>/output/` | ENCORE2 eCLIP datasets with 5000-series IDs (e.g. `QKI_K562_5007`) |
| `/encore/eclip/Skipper/<version>/<run>/output/` | ENCORE eCLIP datasets with 4000-series IDs |
| `/summary_data/` | summary tables (QC, window counts) |

Inside each `output/` folder, files are grouped by type, then named by dataset:

| Subfolder | Files |
| --- | --- |
| `reproducible_enriched_windows/` | `<dataset>.reproducible_enriched_windows.tsv.gz`, the enriched windows |
| `reproducible_enriched_re/` | enriched repetitive elements |
| `secondary_results/bams/dedup/genome_R2/` | deduplicated, read-2-only genome BAMs (ENCODE3) |
| `secondary_results/bams/dedup/genome/` | deduplicated genome BAMs (ENCORE) |
| `secondary_results/bams/raw/genome/` | alignments before deduplication |
| `secondary_results/bigwigs/scaled/plus/`, `.../minus/` | strand-specific signal tracks |
| `QC/`, `gene_sets/`, `homer/` | QC reports, gene set enrichment, motifs |

**Three ways to get files.**

1. **One dataset: BDBag.** Each dataset page on rbp-ark.com has a **Download BDBag** link. The bag's `fetch.txt` lists every file of that dataset with its HTTPS URL, size and path, and `manifest-sha256.txt` its checksum. Notebook 04 reads the same file list from the RBP-ARK API and downloads only the BAMs it needs.
2. **Many datasets: Globus web app.** Sign in at [app.globus.org](https://app.globus.org) with your institution or a free Globus ID, open the RBP-ARK collection (link above), select folders or files, and transfer them to a collection you can write to: your laptop via [Globus Connect Personal](https://www.globus.org/globus-connect-personal), or your institution's cluster or storage collection.
3. **Scripted: Globus CLI.**

```bash
pip install globus-cli
globus login                                   # once; opens a browser
RBPARK=9cd9be17-cbed-4b34-ab76-704b2fd6ac7b
OUT=/encode3/eclip/Skipper/2.0.0/v2/output

# list what is there
globus ls "$RBPARK:$OUT/secondary_results/bams/dedup/genome_R2/" | grep QKI

# copy one file to this computer (Globus Connect Personal must be running)
DEST=$(globus endpoint local-id)
globus transfer "$RBPARK:$OUT/secondary_results/bams/dedup/genome_R2/QKI_K562_ENCSR366YOG_IP_1.genome.Aligned.sort.dedup.R2.bam" \
    "$DEST:$HOME/rbpark/QKI_K562_ENCSR366YOG_IP_1.genome.Aligned.sort.dedup.R2.bam" --label "QKI eCLIP"

# or copy a whole folder
globus transfer --recursive "$RBPARK:$OUT/reproducible_enriched_windows/" "$DEST:$HOME/rbpark/windows/"
globus task list                               # follow progress
```

If the CLI asks for extra consent the first time it touches the collection, it prints the exact `globus session consent ...` command to run. The Globus CLI commands are shown for reference and were not run in testing; the BDBag and HTTPS routes were. Please copy only what you need: the public HTTPS endpoint is shared, and bulk sweeps have taken it offline.

**In this notebook:** RBP-ARK's experiment catalog and windows table come from its data API (`/api/downloads/`), not from the Globus collection.

In [ ]:
import json
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, ".")
import cs3_utils as U

ROOT, DATA = U.folders()
CACHE = DATA / "downloads" / "screen"
CACHE.mkdir(parents=True, exist_ok=True)
print("RBP-ARK API:", U.RBPARK, "| case study folder:", ROOT, "| data folder:", DATA)

## 1. Candidates

In [ ]:
cat = U.rbpark_experiments(CACHE)
encore = cat[cat["project"].str.startswith("ENCORE") & (cat["status"] == "PASS")].copy()
encore["target"] = encore["rbp"].map(lambda r: U.ALIASES.get(r, r))
rmats_files = U.knockdown_files("differential splicing quantifications")
with_kd = encore[encore["target"].isin(set(rmats_files["target"]))]
print(len(encore), "PASS ENCORE/ENCORE2 experiments;", len(with_kd), "with ENCODE knockdown rMATS for their RBP")

In [ ]:
# The RBPs screened (splicing regulators plus others with many windows); PTBP1 for comparison
RBPS = ["QKI", "TARDBP", "SRSF1", "CELF1", "HNRNPK", "KHSRP", "FUS", "RBM15", "SFPQ", "NONO", "PTBP1",
        "TAF15", "IGF2BP1", "FXR1", "PABPC1", "EIF4G2", "UPF1", "ADAR"]
with_kd[with_kd["target"].isin(RBPS)][["subject", "project", "enriched_windows"]]

## 2. Knockdown events changed in both cell lines

FDR < 0.05, |ΔPSI| ≥ 0.1 (ΔPSI = knockdown − control), ≥ 30 junction reads, same direction in K562 and HepG2. This screen uses ENCODE's released rMATS tables; notebook 02 reruns rMATS for QKI.

In [ ]:
KEY = U.SE_COLS[1:]          # chr, strand and the six coordinates
shared, n_events = [], {}
for rbp in RBPS:
    per_cell = {}
    for cell in ("K562", "HepG2"):
        acc, se = U.se_for(rbp, cell, rmats_files, CACHE / "rmats")
        if se is None:
            continue
        se = U.add_se_stats(se)
        per_cell[cell] = se[(se["FDR"] < 0.05) & (se["dpsi"].abs() >= 0.1) & (se["junction_reads"] >= 30)]
    n_events[rbp] = {c: len(d) for c, d in per_cell.items()}
    if len(per_cell) == 2:
        m = per_cell["K562"].merge(per_cell["HepG2"], on=["geneSymbol"] + KEY, suffixes=("_K562", "_HepG2"))
        m = m[np.sign(m["dpsi_K562"]) == np.sign(m["dpsi_HepG2"])]
        m.insert(0, "rbp", rbp)
        shared.append(m)
shared = pd.concat(shared, ignore_index=True)
n_events = pd.DataFrame(n_events).T.assign(shared=shared.groupby("rbp").size()).fillna(0).astype(int)
n_events

## 3. Lift to GRCh38 (UCSC chain)

In [ ]:
lifted = U.lift_events(shared, CACHE)
print(len(shared), "events,", len(lifted), "lifted with unchanged exon length")

## 4. Windows of the same RBP within 500 nt of each exon

A window counts when q ≤ 0.05 and mean log2 enrichment ≥ 2 in a PASS experiment. `rel_nt`: position of the window's center relative to the exon in transcript orientation (negative = upstream intron, 0 = in the exon, positive = downstream intron).

In [ ]:
subjects_by_rbp = {}
for _, x in cat.iterrows():
    subjects_by_rbp.setdefault(U.ALIASES.get(x["rbp"], x["rbp"]), []).append(x["subject"])
rbp_names = {x["rbp"] for _, x in cat.iterrows() if U.ALIASES.get(x["rbp"], x["rbp"]) in RBPS}
windows = U.rbpark_windows(CACHE, rbps=rbp_names)
windows = windows.merge(cat[["subject", "status", "project"]], on="subject", how="left")
bound = windows[(windows["q_value"] <= 0.05) & (windows["log2_enrichment"] >= 2) & (windows["status"] == "PASS")]
print(len(windows), "windows for these RBPs;", len(bound), "bound in PASS experiments")


def rel_nt(strand, es, ee, s, e):
    """Returns the distance in nt from exon `es`-`ee` to the middle of window `s`-`e`, in the transcript's direction
    (`strand`): negative upstream, positive downstream, 0 inside the exon.
    """
    mid = (s + e) / 2
    if strand == "+":
        return -(es - mid) if mid < es else (mid - ee if mid > ee else 0)
    return -(mid - ee) if mid > ee else (es - mid if mid < es else 0)


rows = []
by_key = {k: g for k, g in bound.groupby(["chr", "strand"])}
for _, ev in lifted.iterrows():
    es, ee = ev["exonStart_0base_hg38"], ev["exonEnd_hg38"]
    g = by_key.get((ev["chr"], ev["strand"]))
    hits = pd.DataFrame() if g is None else g[(g["end"] > es - 500) & (g["start"] < ee + 500)
                                               & g["subject"].isin(subjects_by_rbp.get(ev["rbp"], []))]
    best = hits.sort_values("log2_enrichment").groupby("subject").tail(1)
    rows.append({"rbp": ev["rbp"], "gene": ev["geneSymbol"], "exon_hg38": f'{ev["chr"]}:{es}-{ee}({ev["strand"]})',
                 "dpsi_K562": ev["dpsi_K562"], "dpsi_HepG2": ev["dpsi_HepG2"],
                 "n_bound": len(best), "n_bound_encore": int(best["project"].str.startswith("ENCORE").sum()),
                 "max_log2": best["log2_enrichment"].max() if len(best) else np.nan,
                 "windows": "; ".join(f'{r.subject}:{r.log2_enrichment:.1f}@{rel_nt(ev["strand"], es, ee, r.start, r.end):+.0f}'
                                      for r in best.itertuples())})
screen = pd.DataFrame(rows)
screen["min_abs_dpsi"] = screen[["dpsi_K562", "dpsi_HepG2"]].abs().min(axis=1)
ranked = (screen[screen["n_bound"] > 0]
          .sort_values(["n_bound_encore", "n_bound", "min_abs_dpsi"], ascending=False)
          .drop_duplicates(["rbp", "exon_hg38"]))
ranked.to_csv(ROOT / "results" / "screen_candidates.tsv", sep="\t", index=False)
pd.set_option("display.max_colwidth", 120)
ranked.head(40)

**Expected (5 Nov 2026).** PTBP1 dominates the top of the list (133 shared events). The chosen example is the next tier: QKI at PICALM `chr11:85990249-85990399(-)`, with QKI windows ~50 nt upstream of the exon (ENCORE2 4.1, ENCODE3 K562 3.3), and QKI downstream of a second PICALM exon (24 nt). Other RBPs in the table are not followed up.

## 5. Panel b: PICALM exon 13 and the 24-nt exon

In [ ]:
CHR, STRAND = "chr11", "-"
EX13, EX24 = U.PICALM_GRCH38["exon13"][2:], U.PICALM_GRCH38["exon24nt"][2:]     # GRCh38, 0-based half-open
loc = windows[(windows["chr"] == CHR) & (windows["strand"] == STRAND) & windows["subject"].isin(list(U.ECLIP))
              & (windows["end"] > 85975000) & (windows["start"] < 85995000)].sort_values(["subject", "start"])
loc["bound"] = (loc["q_value"] <= 0.05) & (loc["log2_enrichment"] >= 2)
near13 = loc[(loc["end"] > EX13[0] - 200) & (loc["start"] < EX13[1] + 200)]
near13[["subject", "start", "end", "feature_type", "log2_enrichment", "q_value", "bound"]]

In [ ]:
near24 = loc[(loc["end"] > EX24[0] - 200) & (loc["start"] < EX24[1] + 200)]
near24[["subject", "start", "end", "feature_type", "log2_enrichment", "q_value", "bound"]]

Expected: `85990399-85990499` (0-100 nt upstream of exon 13, minus strand): QKI ENCORE2 4.08 (q 7.5e-11), ENCODE3 K562 3.31; `85977969-85978068` (0-100 nt downstream of the 24-nt exon): QKI ENCORE2 4.59, HepG2 4.28.

In [ ]:
seqs = {"exon13_upstream_0_100": U.ucsc_sequence(CHR, EX13[1], EX13[1] + 100, STRAND),
        "exon24nt_downstream_0_100": U.ucsc_sequence(CHR, EX24[0] - 100, EX24[0], STRAND)}
for k, s in seqs.items():
    print(f"{k:26s} ACUAAY: {len(re.findall('ACUAA[CU]', s))}  UAAY: {len(re.findall('UAA[CU]', s))}  U: {s.count('U') / len(s):.0%}")
    print("  ", s)

The exon 13 upstream window holds `ACUAAUUACAAUACUAAUUU`: two ACUAAU elements 10 nt apart, the bipartite QKI response element.

Transcripts come from **GENCODE v40**, the annotation used in all the case studies (the 47 MB GTF is downloaded once).

In [ ]:
tx = U.gencode_transcripts("PICALM", CACHE)
rows = []
for tid, t in tx.items():
    ex = [tuple(e) for e in t["exons"]]
    spans13 = ex[0][0] < EX13[0] and ex[-1][1] > EX13[1]
    rows.append({"transcript": tid, "name": t["name"], "type": t["type"], "MANE": "MANE_Select" in t["tags"],
                 "exon 13 (150 nt)": tuple(EX13) in ex, "skips exon 13": spans13 and not any(s < EX13[1] and e > EX13[0] for s, e in ex),
                 "24-nt exon": tuple(EX24) in ex})
tx_table = pd.DataFrame(rows).sort_values(["type", "name"])
print(tx_table.groupby("type")[["exon 13 (150 nt)", "skips exon 13", "24-nt exon"]].sum())
tx_table[tx_table["type"] == "protein_coding"]

In [ ]:
out = {"chr": CHR, "strand": STRAND, "exons": {"exon13": list(EX13), "exon24nt": list(EX24)}, "annotation": "GENCODE v40",
       "transcripts": tx,
       "windows": loc[["subject", "start", "end", "log2_enrichment", "q_value"]].values.tolist(), "rna": seqs}
(ROOT / "figures" / "data" / "picalm_locus.json").write_text(json.dumps(out, indent=1, default=float))
print("wrote", ROOT / "figures" / "data" / "picalm_locus.json")